In [ ]:
# pip install langgraph langchain-core


In [ ]:
import json

def _safe_json(obj, max_len=8000):
    try:
        s = json.dumps(obj, default=str)
        if len(s) > max_len:
            return s[:max_len] + "...(truncated)"
        return s
    except Exception:
        return str(obj)


def record_decision(
    state: dict,
    *,
    agent: str,
    type: str,
    context_facts: dict,
    commitment_action: dict,
    rules_fired: list[str] | None = None,
    alternatives: list[str] | None = None,
    confidence: float | None = None,
    evidence: list[dict] | None = None,
    status: str = "committed",
):
    decision = {
        "id": str(uuid.uuid4()),
        "agent": agent,
        "type": type,
        "context": {
            "facts": context_facts,
            "constraints": {},
            "assumptions": {},
        },
        "evidence": evidence or [],
        "commitment": {
            "action": commitment_action,
            "reversible": True,
        },
        "rule_applications": [
            {"rule_id": r, "status": "triggered"}
            for r in (rules_fired or [])
            if r is not None
        ],
        "alternatives": alternatives or [],
        "confidence": confidence,
        "status": status,
        "timestamp": _now(),
    }

    state.setdefault("decisions", []).append(decision)

    # 🔍 TELEMETRY EXPORT
    span = trace.get_current_span()
    if span and span.is_recording():

        # --- Core decision identity ---
        span.set_attribute("decision.id", decision["id"])
        span.set_attribute("decision.agent", agent)
        span.set_attribute("decision.type", type)
        span.set_attribute("decision.status", status)

        if confidence is not None:
            span.set_attribute("decision.confidence", confidence)

        # --- Rule analytics (flat for querying) ---
        span.set_attribute("decision.rule_count", len(decision["rule_applications"]))
        for rule in decision["rule_applications"]:
            span.set_attribute(f"decision.rule.{rule['rule_id']}", True)

        # --- Alternatives ---
        span.set_attribute("decision.alternative_count", len(decision["alternatives"]))
        for alt in decision["alternatives"]:
            span.set_attribute(f"decision.alternative.{alt}", True)

        # --- Context facts (flattened for metrics) ---
        for k, v in context_facts.items():
            if isinstance(v, (str, int, float, bool)):
                span.set_attribute(f"decision.context.{k}", v)

        # --- JSON payloads for deep inspection ---
        span.set_attribute(
            "decision.context_json",
            _safe_json(decision["context"])
        )
        span.set_attribute(
            "decision.commitment_json",
            _safe_json(decision["commitment"]["action"])
        )
        span.set_attribute(
            "decision.evidence_json",
            _safe_json(decision["evidence"])
        )

    return decision


In [ ]:
# interpretability.py
from datetime import datetime
import uuid


def _now():
    return datetime.utcnow().isoformat()

In [ ]:
class InterpretabilityViolation(RuntimeError):
    pass


def enforce_decision_recording(agent_name: str, state: dict, before_count: int):
    after_count = len(state.get("decisions", []))
    if after_count == before_count:
        raise InterpretabilityViolation(
            f"Agent '{agent_name}' exited without recording a decision"
        )


In [ ]:
from langgraph.errors import GraphInterrupt

In [ ]:
from functools import wraps
from typing import Callable, Optional
from opentelemetry import trace, context
from opentelemetry.trace import SpanKind

tracer = trace.get_tracer("pm-agent")


def agent_span(
    name: Optional[str] = None,
    record_inputs: bool = True,
    record_outputs: bool = True,
):
    def decorator(fn: Callable):
        span_name = name or fn.__name__
        import time
        time.sleep(5) #0.005)  # 5ms


        @wraps(fn)
        def wrapper(state: dict, *args, **kwargs):
          with tracer.start_as_current_span(span_name) as span:
            # 🔹 Capture whatever context LangGraph is running with
            parent_ctx = context.get_current()

            # 🔹 Explicitly start span WITH parent context
            span = tracer.start_span(
                span_name,
                context=parent_ctx,
                kind=SpanKind.INTERNAL,
            )

            # 🔹 Force this span into the active context
            token = context.attach(
                trace.set_span_in_context(span, parent_ctx)
            )

            try:
                # --- Inputs ---
                if record_inputs and isinstance(state, dict):
                    for k, v in state.items():
                        if k in {"issues", "llm_traces"} and hasattr(v, "__len__"):
                            span.set_attribute(f"input.{k}.count", len(v))
                        elif isinstance(v, (str, int, float, bool)):
                            span.set_attribute(f"input.{k}", v)


                decision_count_before = len(state.get("decisions", []))
                span.set_attribute("agent.name", span_name)


                result = fn(state, *args, **kwargs)


                enforce_decision_recording(
                    span_name,
                    result,
                    decision_count_before,
                )


                # --- Outputs ---
                if record_outputs and isinstance(result, dict):
                    for k, v in result.items():
                        if isinstance(v, (str, int, float, bool)):
                            span.set_attribute(f"output.{k}", v)

                return result

            except GraphInterrupt:
                span.end()
                context.detach(token)
                raise

            except Exception as e:
                # 🔥 Ensure failures are visible in Honeycomb
                span.record_exception(e)
                span.set_status(trace.Status(trace.StatusCode.ERROR))
                raise

            finally:
                # 🔹 Clean teardown (critical)
                # enforce_decision_recording(span_name, state, decision_count_before)

                span.end()
                context.detach(token)

        return wrapper

    return decorator


In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter

# trace.set_tracer_provider(TracerProvider())
from opentelemetry.sdk.resources import Resource, SERVICE_NAME
from opentelemetry.sdk.trace import TracerProvider

resource = Resource.create({
    SERVICE_NAME: "pm-agent",   # 👈 this is what Honeycomb shows
    "service.namespace": "langgraph",
    "deployment.environment": "local",  # optional but useful
})

trace.set_tracer_provider(
    TracerProvider(resource=resource)
)



# https://api.eu1.honeycomb.io:443
span_processor = BatchSpanProcessor(
    OTLPSpanExporter(
        endpoint="https://api.eu1.honeycomb.io:443/v1/traces",
        headers={
            "x-honeycomb-team": "xxxx",
            "x-honeycomb-dataset": "pm-agent-traces"
        }
    )
)

trace.get_tracer_provider().add_span_processor(span_processor)

tracer = trace.get_tracer("pm-agent")


In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class JiraIssue:
    key: str
    story_points: int
    status: str
    type: str
    blocked: bool = False
    dependencies: List[str] = field(default_factory=list)
    days_delayed: int = 0




In [ ]:
# Adjust velocity based on new team members
def velocity_adjustment_tool(base_velocity, new_members):
    return base_velocity + 2 * new_members  # simple heuristic

# Analyze dependencies
def dependency_analyzer_tool(issues):
    dep_map = {}
    for issue in issues:
        deps = issue.dependencies if hasattr(issue, "dependencies") else issue.get("dependencies", [])
        if deps:
            dep_map[issue.key if hasattr(issue, "key") else issue["key"]] = deps
    return dep_map



In [ ]:
# from otel_decorator import agent_span

@agent_span()
def jira_data_agent(state: dict) -> dict:
    state.setdefault("issues", [])
    state["issues"] = [
        JiraIssue("PAY-101", 13, "To Do", "feature"),
        JiraIssue("AUTH-44", 5, "In Progress", "enhancement"),
        JiraIssue("BUG-17", 3, "To Do", "bugfix"),
        JiraIssue("API-88", 21, "To Do", "feature", blocked=True, days_delayed=12),
        JiraIssue("OPS-9", 8, "To Do", "enhancement", dependencies=["API-88"]),
    ]
    state.setdefault("risks", {})

    # Debug log
    print("[JIRA DATA AGENT] Issues loaded:")
    for i in state["issues"]:
        key = i.key if hasattr(i, "key") else i.get("key")
        story_points = i.story_points if hasattr(i, "story_points") else i.get("story_points")
        blocked = i.blocked if hasattr(i, "blocked") else i.get("blocked", False)
        print(f" - {key}, blocked={blocked}")
        print(f" - {story_points}")
        record_decision(
            state,
            agent="jira_data_agent",
            type="load_issues",

            context_facts={
                "source": "jira",
            },

            commitment_action={
                "loaded_issues": len(state["issues"]),
            },

            rules_fired=[],
            alternatives=["manual_entry"],
            confidence=0.99,

            evidence=[
                {"source": "jira_api", "entity": "issues"}
            ],
        )

    state["next_agent"] = "estimation"

    return state


In [ ]:
# from otel_decorator import agent_span

@agent_span()
def estimation_agent(state: dict) -> dict:
    state.setdefault("estimation_flags", [])

    base_velocity = state.get("team_velocity", 13)
    new_members = state.get("new_members", 0)
    adjusted_velocity = velocity_adjustment_tool(base_velocity, new_members)
    state["adjusted_velocity"] = adjusted_velocity

    flags = []

    for issue in state.get("issues", []):
        # EXPLICIT TYPE HANDLING — NO getattr fallback
        if isinstance(issue, dict):
            points = issue.get("story_points", 0)
            key = issue.get("key")
        else:
            points = issue.story_points
            key = issue.key

        if points > adjusted_velocity:
            flags.append({
                "issue": key,
                "points": points,
                "velocity": adjusted_velocity,
                "recommendation": "Split user story"
            })

    state["estimation_flags"] = flags

    print("[ESTIMATION AGENT]", flags)
    record_decision(
        state,
        agent="estimation_agent",
        type="velocity_adjustment",

        context_facts={
            "base_velocity": base_velocity,
            "new_members": new_members,
        },

        commitment_action={
            "adjusted_velocity": adjusted_velocity,
            "flagged_issues": len(flags),
        },

        rules_fired=["velocity_adjustment_heuristic"],
        alternatives=["static_velocity"],
        confidence=0.9,
    )

    state["next_agent"] = "defect_alignment"
    return state

In [ ]:
# pip install langchain

In [ ]:
# pip install langchain_openai

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.messages import SystemMessage, HumanMessage

import os

os.environ["OPENAI_API_KEY"] = "xxxxxx"
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)



In [ ]:
def defect_risk_assessment_tool(aligned_defects):
    with tracer.start_as_current_span(
        "defect_risk_assessment_tool",
        kind=SpanKind.CLIENT
    ) as span:

        span.set_attribute("tool.name", "defect_risk_assessment")

        avg_similarity = (
            sum(d["similarity_score"] for d in aligned_defects)
            / len(aligned_defects)
        ) if aligned_defects else 1.0

        risk_score = 1 - avg_similarity
        span.set_attribute("output.risk_score", risk_score)

        return {"risk_score": risk_score}


In [ ]:
def sprint_capacity_analysis_tool(team_size, sprint_days):
    with tracer.start_as_current_span(
        "sprint_capacity_analysis_tool",
        kind=SpanKind.CLIENT
    ) as span:

        span.set_attribute("tool.name", "sprint_capacity_analysis")

        capacity = team_size * sprint_days * 6  # 6 productive hrs/day
        span.set_attribute("output.capacity_hours", capacity)

        return {"capacity_hours": capacity}


In [ ]:
def historical_velocity_lookup_tool(last_n_sprints):
    with tracer.start_as_current_span(
        "historical_velocity_lookup_tool",
        kind=SpanKind.CLIENT
    ) as span:

        span.set_attribute("tool.name", "historical_velocity_lookup")

        avg_velocity = 32  # mock
        span.set_attribute("output.avg_velocity", avg_velocity)

        return {"avg_velocity": avg_velocity}


In [ ]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "defect_risk_assessment_tool",
            "description": "Assess sprint delivery risk from aligned defects",
            "parameters": {
                "type": "object",
                "properties": {
                    "aligned_defects": {
                        "type": "array",
                        "items": {"type": "object"}
                    }
                },
                "required": ["aligned_defects"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "sprint_capacity_analysis_tool",
            "description": "Estimate total sprint capacity",
            "parameters": {
                "type": "object",
                "properties": {
                    "team_size": {"type": "integer"},
                    "sprint_days": {"type": "integer"}
                },
                "required": ["team_size", "sprint_days"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "historical_velocity_lookup_tool",
            "description": "Fetch average velocity from recent sprints",
            "parameters": {
                "type": "object",
                "properties": {
                    "last_n_sprints": {"type": "integer"}
                },
                "required": ["last_n_sprints"]
            }
        }
    }
]


In [ ]:
# old defect alignment with tools
@agent_span()
def defect_backlog_semantic_alignment_agent(state: dict) -> dict:
    current_issues = state.get("current_issues", [])
    historical_defects = state.get("historical_defects", [])

    aligned_defects = [
        {
            "id": d["id"],
            "matched_issue": current_issues[0]["id"],
            "similarity_score": 0.6
        }
        for d in historical_defects
    ]

    system_prompt = """
You are a Scrum planning assistant.

You may call tools if needed:
- defect_risk_assessment_tool
- sprint_capacity_analysis_tool
- historical_velocity_lookup_tool

Call only the tools that are necessary.
You may call multiple tools.
"""

    user_prompt = f"""
Aligned defects:
{json.dumps(aligned_defects)}

Team size: {state.get("team_size", 5)}
Sprint days: {state.get("sprint_days", 10)}
"""
    llm_with_tools = llm.bind_tools(tools_schema)

    response = llm_with_tools.invoke([
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
    ])

    if response.tool_calls:
        for tool_call in response.tool_calls:
            name = tool_call["name"]
            args = tool_call["args"]

    # response = llm.chat.completions.create(
    #     model="gpt-4o-mini",
    #     messages=[
    #         {"role": "system", "content": system_prompt},
    #         {"role": "user", "content": user_prompt}
    #     ],
    #     tools=tools_schema,
    #     tool_choice="auto",
    #     temperature=0
    # )

    # message = response.choices[0].message
            tool_results = {}

    # if response.tool_calls:

    # if message.tool_calls:
        # for   tool_call in response.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)

            if name == "defect_risk_assessment_tool":
                tool_results.update(
                    defect_risk_assessment_tool(**args)
                )

            elif name == "sprint_capacity_analysis_tool":
                tool_results.update(
                    sprint_capacity_analysis_tool(**args)
                )

            elif name == "historical_velocity_lookup_tool":
                tool_results.update(
                    historical_velocity_lookup_tool(**args)
                )

    state.update(tool_results)

    record_decision(
        state,
        agent="defect_backlog_semantic_alignment_agent",
        type="scrum_analysis_with_tool_orchestration",
        context_facts={
            "aligned_count": len(aligned_defects),
            "tools_invoked": list(tool_results.keys())
        },
        commitment_action=tool_results,
        rules_fired=["llm_multi_tool_orchestration"],
        alternatives=["no_tool_usage"],
        confidence=0.85,
        status="committed"
    )

    state["next_agent"] = "unplanned_work_prediction"
    return state


In [ ]:
# new defect alignment with tools
@agent_span()
def defect_backlog_semantic_alignment_agent(state: dict) -> dict:
  try:
      current_issues = state.get("issues", []) # Changed from "current_issues" to "issues"
      print(f"DEBUG(defect_alignment): current_issues at start: {current_issues}")
      historical_defects = state.get("historical_defects", [])

      # If no historical defects exist, create synthetic ones for demo
      if not historical_defects:
          historical_defects = [
              {"id": "D-1", "title": "Payment timeout error", "module": "payments"},
              {"id": "D-2", "title": "Auth token expiration bug", "module": "auth"},
              {"id": "D-3", "title": "API latency spike", "module": "api"},
              {"id": "D-4", "title": "Null pointer in payment service", "module": "payments"},
          ]
      state["historical_defects"] = historical_defects

      # --- 1️⃣ Human-in-the-loop: approve defects (notebook-friendly) ---
      approved_ids = []
      if historical_defects:
          print("\nCurrent historical defects:")
          for d in historical_defects:
              print(f"  ID: {d['id']}, description: {d.get('description', '')}")

          while True:
              try:
                  print("About to call input()")
                  approved_ids_input = input(
                      "Enter comma-separated IDs of defects to approve for alignment: "
                  )
                  print("Input returned:", approved_ids_input)
                  approved_ids = [id_.strip() for id_ in approved_ids_input.split(",") if id_.strip()]
                  break
              except Exception as e:
                  print(f"Invalid input, try again: {e}")

      # --- 2️⃣ Filter aligned defects based on human approval ---
      aligned_defects = []
      # Only proceed with alignment if current_issues is not empty and there are approved_ids
      if current_issues and approved_ids and len(current_issues) > 0:
          print(f"DEBUG(defect_alignment): current_issues has {len(current_issues)} items. Proceeding with alignment.")
          issue_to_match_key = current_issues[0].key # This will now be safe due to the len check
          aligned_defects = [
              {"id": d["id"], "matched_issue": issue_to_match_key, "similarity_score": 0.6}
              for d in historical_defects if d["id"] in approved_ids
          ]
      else:
          print("DEBUG(defect_alignment): Skipping defect alignment. Either current_issues is empty, no approved defects, or current_issues has 0 length.")

      approved_count = len(aligned_defects)

      tool_results = {}
      tools_called = []

      if approved_count > 0:
          # --- 3️⃣ LLM prompt for tool orchestration ---
          system_prompt = """
          You are a Scrum planning assistant.

          You may call tools if needed:
          - defect_risk_assessment_tool
          - sprint_capacity_analysis_tool
          - historical_velocity_lookup_tool

          Call only the tools that are necessary.
          You may call multiple tools.
          """
          user_prompt = f"""
          Aligned defects (approved by human):
          {json.dumps(aligned_defects)}

          Team size: {state.get("team_size", 5)}
          Sprint days: {state.get("sprint_days", 10)}
          """

          llm_with_tools = llm.bind_tools(tools_schema)

          response = llm_with_tools.invoke([
              {"role": "system", "content": system_prompt},
              {"role": "user", "content": user_prompt},
          ])
          print("DEBUG(defect_alignment): LLM response:", response)
          print("DEBUG(defect_alignment): LLM response.tool_calls:", response.tool_calls)
          # --- 4️⃣ Execute any tools the LLM decided to call ---
          if response.tool_calls:
              for tool_call in response.tool_calls:
                  print("DEBUG(defect_alignment): Tool call:", tool_call)
                  # name = tool_call.function.name
                  name = tool_call['name']
                  # args = json.loads(tool_call.function.arguments)
                  args = tool_call.get('args', {})

                  if name == "defect_risk_assessment_tool":
                      # Ensure aligned_defects is passed explicitly as it's required by the tool
                      # and LLM might not always include it in args
                      if "aligned_defects" not in args:
                          args["aligned_defects"] = aligned_defects
                      tool_results.update(defect_risk_assessment_tool(**args))
                  elif name == "sprint_capacity_analysis_tool":
                      # sprint_capacity_analysis_tool does not take aligned_defects
                      tool_results.update(sprint_capacity_analysis_tool(**args))
                  elif name == "historical_velocity_lookup_tool":
                      # historical_velocity_lookup_tool does not take aligned_defects
                      tool_results.update(historical_velocity_lookup_tool(**args))

                  tools_called.append(name)

          # --- 5️⃣ Ensure at least one tool is called ---
          if not tools_called and approved_count > 0: # Only call if there are approved defects
              tool_results.update(defect_risk_assessment_tool(aligned_defects=aligned_defects))
              tools_called.append("defect_risk_assessment_tool")

          state.update(tool_results)

      # --- 6️⃣ Always record decision ---
      record_decision(
          state,
          agent="defect_backlog_semantic_alignment_agent",
          type="scrum_analysis_with_tool_orchestration",
          context_facts={
              "aligned_count": len(aligned_defects),
              "approved_count": approved_count,
              "tools_invoked": tools_called,
          },
          commitment_action={"tool_results": tool_results},
          rules_fired=["llm_multi_tool_orchestration"],
          alternatives=["no_tool_usage", "manual_planning_review"],
          confidence=0.85,
          status="committed",
      )
  except Exception as e:
        # ALWAYS record failure
        record_decision(
            state,
            agent="defect_backlog_semantic_alignment_agent",
            type="scrum_analysis_with_tool_orchestration",
            context_facts={"error": str(e)},
            commitment_action={},
            rules_fired=["exception_path"],
            alternatives=["retry", "manual_review"],
            confidence=0.2,
            status="failed",
        )
        raise

  state["next_agent"] = "unplanned_work_prediction"

  return state

In [ ]:
from unittest.mock import patch
import json

# --- Mock objects ---
class MockIssue:
    def __init__(self, key):
        self.key = key

# --- Mock tools (stubs) ---
def defect_risk_assessment_tool(**kwargs):
    return {"defect_risk_results": "mocked"}

def sprint_capacity_analysis_tool(**kwargs):
    return {"sprint_capacity_results": "mocked"}

def historical_velocity_lookup_tool(**kwargs):
    return {"historical_velocity_results": "mocked"}

# --- Mock LLM ---
class MockLLM:
    def bind_tools(self, tools_schema):
        return self
    def invoke(self, prompts):
        # Simulate a tool call
        class Response:
            tool_calls = [
                type("ToolCall", (), {
                    "function": type("Func", (), {
                        "name": "defect_risk_assessment_tool",
                        "arguments": json.dumps({})
                    })()
                })()
            ]
        return Response()

llm = MockLLM()
tools_schema = {}  # dummy

# --- Mock record_decision (append to state) ---
def mock_record_decision(state, **kwargs):
    if "decisions" not in state:
        state["decisions"] = []
    state["decisions"].append({
        "agent": kwargs.get("agent"),
        "status": kwargs.get("status"),
        "context_facts": kwargs.get("context_facts", {}),
    })

# --- Prepare state ---
state = {
    "issues": [MockIssue("JIRA-123")],
    "historical_defects": [
        {"id": "D-1", "title": "Payment error", "module": "payments"},
        {"id": "D-2", "title": "Auth bug", "module": "auth"},
    ],
    "team_size": 5,
    "sprint_days": 10
}

# --- Patch input and record_decision in notebook globals ---
with patch("builtins.input", return_value="D-1,D-2"):
    # patch the global function directly
    globals()['record_decision'] = mock_record_decision
    updated_state = defect_backlog_semantic_alignment_agent(state)

# --- Verify results ---
print("Decisions recorded:", updated_state.get("decisions"))
print("Next agent:", updated_state.get("next_agent"))

DEBUG(defect_alignment): current_issues at start: [<__main__.MockIssue object at 0x78b86fe2f1d0>]

Current historical defects:
  ID: D-1, description: 
  ID: D-2, description: 
About to call input()
Input returned: D-1,D-2
DEBUG(defect_alignment): current_issues has 1 items. Proceeding with alignment.
Decisions recorded: [{'agent': 'defect_backlog_semantic_alignment_agent', 'status': 'committed', 'context_facts': {'aligned_count': 2, 'approved_count': 2, 'tools_invoked': ['defect_risk_assessment_tool']}}]
Next agent: unplanned_work_prediction


In [ ]:
# old one without tools
@agent_span()
def defect_backlog_semantic_alignment_agent(state: dict) -> dict:
    # --- Ensure required state keys exist ---
    current_issues = state.get("issues", [])
    historical_defects = state.setdefault("historical_defects", [])

    # If no historical defects exist, create synthetic ones for demo
    if not historical_defects:
        historical_defects = [
            {"id": "D-1", "title": "Payment timeout error", "module": "payments"},
            {"id": "D-2", "title": "Auth token expiration bug", "module": "auth"},
            {"id": "D-3", "title": "API latency spike", "module": "api"},
            {"id": "D-4", "title": "Null pointer in payment service", "module": "payments"},
        ]
        state["historical_defects"] = historical_defects

    # --- Normalize issue format ---
    normalized_issues = []
    for issue in current_issues:
        if isinstance(issue, dict):
            normalized_issues.append(issue)
        else:
            normalized_issues.append({
                "key": issue.key,
                "type": issue.type,
                "story_points": issue.story_points,
                "blocked": issue.blocked,
            })

    aligned_defects = []
    alignment_details = []

    # --- Simple semantic similarity heuristic ---
    for defect in historical_defects:
        best_score = 0.0
        best_issue_key = None

        defect_text = defect.get("title", "").lower()
        defect_module = defect.get("module", "").lower()

        for issue in normalized_issues:
            issue_key = issue.get("key", "")
            issue_type = issue.get("type", "").lower()

            # Basic keyword overlap heuristic
            keyword_score = 0.0
            if "payment" in defect_text and "pay" in issue_key.lower():
                keyword_score += 0.4
            if "auth" in defect_text and "auth" in issue_key.lower():
                keyword_score += 0.4
            if "api" in defect_text and "api" in issue_key.lower():
                keyword_score += 0.4

            # Module affinity boost
            module_boost = 0.2 if defect_module in issue_key.lower() else 0.0

            similarity_score = min(1.0, keyword_score + module_boost)

            if similarity_score > best_score:
                best_score = similarity_score
                best_issue_key = issue_key

        # Apply alignment threshold
        if best_score >= 0.5:
            aligned_defects.append(defect)
            alignment_details.append({
                "id": defect["id"],
                "matched_issue": best_issue_key,
                "similarity_score": round(best_score, 2),
            })

    # --- Determine if human review is required ---
    requires_human = (
        len(aligned_defects) > 10 or
        any(d["similarity_score"] < 0.65 for d in alignment_details)
    )

    decision_status = "proposed" if requires_human else "committed"

    # --- Record interpretability decision ---
    record_decision(
        state,
        agent="defect_backlog_semantic_alignment_agent",
        type="align_defects_by_semantic_relevance",

        context_facts={
            "story_count": len(current_issues),
            "defect_count": len(historical_defects),
            "aligned_count": len(aligned_defects),
        },

        commitment_action={
            "aligned_defect_ids": [d["id"] for d in aligned_defects],
            "alignment_details": alignment_details,
        },

        rules_fired=[
            "semantic_similarity_threshold",
            "module_affinity_boost",
            "human_review_threshold" if requires_human else None,
        ],

        alternatives=[
            "module_only_alignment",
            "manual_triage",
        ],

        confidence=0.75 if not requires_human else 0.6,
        status=decision_status,
    )

    # --- Update state ---
    state["aligned_defects"] = aligned_defects
    state["needs_human_alignment_review"] = requires_human

    # IMPORTANT: Always explicitly set next_agent
    state["next_agent"] = (
        "human_alignment_review"
        if requires_human
        else "unplanned_work_prediction"
    )

    if requires_human:
      state["human_review_task"] = {
          "type": "defect_alignment_review",
          "question": "Approve or modify the proposed defect alignments?",
          "proposed_alignment": alignment_details,
          "options": [
              "approve_all",
              "approve_subset",
              "reject_all",
              "manual_override"
          ],
      }


    return state


In [ ]:
@agent_span()
def unplanned_work_prediction_agent(state: dict) -> dict:
    previous_tasks = state.get("previous_sprint_tasks", [])

    mid_sprint_starts = [
        t for t in previous_tasks
        if t.get("started_mid_sprint", False)
    ]

    ratio = (
        len(mid_sprint_starts) / len(previous_tasks)
        if previous_tasks else 0.0
    )

    # Simple heuristic
    if ratio > 0.3:
        risk_level = "high"
        buffer_pct = 0.25
    elif ratio > 0.15:
        risk_level = "medium"
        buffer_pct = 0.15
    else:
        risk_level = "low"
        buffer_pct = 0.05

    state["unplanned_work_risk"] = {
        "risk_level": risk_level,
        "buffer_pct": buffer_pct,
        "mid_sprint_start_ratio": round(ratio, 2),
    }

    record_decision(
        state,
        agent="unplanned_work_prediction_agent",
        type="predict_unplanned_work",

        context_facts={
            "previous_task_count": len(previous_tasks),
            "mid_sprint_starts": len(mid_sprint_starts),
        },

        commitment_action={
            "risk_level": risk_level,
            "recommended_capacity_buffer": buffer_pct,
        },

        rules_fired=[
            "mid_sprint_start_frequency",
        ],

        alternatives=[
            "fixed_buffer",
            "ignore_unplanned_work",
        ],

        confidence=0.7,
    )

    state["next_agent"] = "sprint_health"
    return state


In [ ]:
@agent_span()
def human_alignment_review_agent(state: dict) -> dict:
    proposed = next(
        d for d in reversed(state["decisions"])
        if d["type"] == "align_defects_by_semantic_relevance"
        and d["status"] == "proposed"
    )

    # --- Display what decision is needed ---
    print("\n=== HUMAN REVIEW REQUIRED ===")
    print("Approve or modify the proposed defect alignments?")
    for d in proposed["commitment"]["action"]["aligned_defect_ids"]:
        print(" -", d)
    print("Similarity details:", proposed["commitment"]["action"]["alignment_details"])

    # --- Notebook-friendly wait for human input ---
    user_input = input(
        "Enter IDs to approve (comma-separated), or type 'all' to approve all: "
    ).strip()

    if user_input.lower() == "all":
        human_selected_defects = proposed["commitment"]["action"]["aligned_defect_ids"]
    else:
        human_selected_defects = [x.strip() for x in user_input.split(",")]

    # --- Record decision after human input ---
    record_decision(
        state,
        agent="human_alignment_reviewer",
        type="confirm_defect_alignment",

        context_facts={
            "proposed_decision_id": proposed["id"],
            "review_reason": "High defect volume",
        },

        commitment_action={
            "approved_defect_ids": human_selected_defects,
        },

        rules_fired=[
            "human_authority_override",
        ],

        confidence=1.0,
        status="committed",
    )

    state["aligned_defects"] = [
        d for d in state.get("historical_defects", [])
        if d["id"] in human_selected_defects
    ]

    state["next_agent"] = "unplanned_work_prediction"
    return state


In [ ]:
# from otel_decorator import agent_span

@agent_span()
def sprint_health_agent(state: dict) -> dict:
    issues = state.get("issues", [])
    state.setdefault("risks", {})

    blocked_issues = []
    for i in issues:
        if isinstance(i, dict):
            blocked = i.get("blocked", False)
            key = i.get("key")
            days_delayed = i.get("days_delayed", 0)
        else:
            blocked = i.blocked
            key = i.key
            days_delayed = i.days_delayed

        if blocked:
            blocked_issues.append((key, days_delayed))

    oversized = len(state.get("estimation_flags", []))
    dependency_map = dependency_analyzer_tool(issues)
    dependent_issues = [d for deps in dependency_map.values() for d in deps]

    health = 1.0
    health -= len(blocked_issues) * 0.2
    health -= oversized * 0.15
    health -= len(dependent_issues) * 0.05

    sprint_duration_days = 10
    feasibility_flags = []
    for key, days_delayed in blocked_issues:
        if days_delayed >= sprint_duration_days:
            feasibility_flags.append({
                "issue": key,
                "days_delayed": days_delayed,
                "message": "Unlikely to complete in this sprint"
            })
            health -= 0.1

    state["sprint_health"] = round(max(0.0, health), 2)
    state["risks"].update({
        "blocked_issues": len(blocked_issues),
        "oversized_stories": oversized,
        "dependent_stories": len(dependent_issues),
        "blocked_dependencies_map": dependency_map,
        "feasibility_flags": feasibility_flags
    })

    print("[SPRINT HEALTH]", state["sprint_health"], state["risks"])
    record_decision(
        state,
        agent="sprint_health_agent",
        type="compute_sprint_health",

        context_facts={
            "blocked_issues": len(blocked_issues),
            "oversized_stories": oversized,
            "dependent_stories": len(dependent_issues),
        },

        commitment_action={
            "sprint_health_score": state["sprint_health"],
            "risk_level": "high" if state["sprint_health"] < 0.6 else "normal",
        },

        rules_fired=[
            "blocked_penalty",
            "oversized_penalty",
            "dependency_penalty",
        ],

        alternatives=["ignore_blockers"],
        confidence=0.85,
    )

    state["next_agent"] = "prioritization"
    return state



In [ ]:
# from otel_decorator import agent_span

@agent_span()
def prioritization_agent(state: dict) -> dict:
    state.setdefault("prioritized_backlog", [])

    focus_weights = state.get(
        "focus_weights",
        {"feature": 0.5, "enhancement": 0.3, "bugfix": 0.2}
    )

    backlog = []

    for issue in state.get("issues", []):
        if isinstance(issue, dict):
            issue_type = issue.get("type", "feature")
            key = issue.get("key")
            points = issue.get("story_points", 0)
        else:
            issue_type = issue.type
            key = issue.key
            points = issue.story_points

        weight = focus_weights.get(issue_type, 0.0)

        # Priority heuristic (policy, not fact)
        priority_score = weight + (1 / max(points, 1))

        backlog.append({
            "key": key,
            "type": issue_type,
            "points": points,
            "score": round(priority_score, 2),
        })

    backlog.sort(key=lambda x: x["score"], reverse=True)

    ordered_keys = [item["key"] for item in backlog]
    state["prioritized_backlog"] = ordered_keys

    print("[PRIORITIZATION AGENT] Ranked backlog:")
    for item in backlog:
        print(" -", item)

    # 🧠 SEMANTIC DECISION
    record_decision(
        state,
        agent="prioritization_agent",
        type="rank_backlog",

        # 🔹 What was true when we decided
        context_facts={
            "focus_weights": focus_weights,
            "issue_count": len(backlog),
        },

        # 🔹 What we committed to
        commitment_action={
            "backlog_order": ordered_keys,
            "scoring_method": "weighted_priority_plus_size_bias",
        },

        # 🔹 Policies / heuristics applied
        rules_fired=[
            "focus_weight_priority",
            "small_story_bias",
        ],

        # 🔹 Legitimate alternatives
        alternatives=[
            "fifo",
            "wsjf",
            "pure_weighted",
        ],

        confidence=0.88,
    )

    state["next_agent"] = "planning"
    return state




In [ ]:
# from otel_decorator import agent_span

@agent_span()
def planning_agent(state: dict) -> dict:
    state.setdefault("plan_decisions", [])

    sprint_health = state.get("sprint_health", 1.0)
    risks = state.get("risks", {})
    estimation_flags = state.get("estimation_flags", [])
    prioritized_backlog = state.get("prioritized_backlog", [])

    decisions = []

    # 1️⃣ Oversized stories → split recommendation
    if estimation_flags:
        decisions.append({
            "action": "Split oversized stories",
            "issues": [f["issue"] for f in estimation_flags],
            "reason": "Story points exceed adjusted team velocity"
        })

    # 2️⃣ Blocked stories that are infeasible this sprint
    feasibility_flags = risks.get("feasibility_flags", [])
    if feasibility_flags:
        decisions.append({
            "action": "Move blocked stories out of sprint",
            "issues": [f["issue"] for f in feasibility_flags],
            "reason": "Blocked long enough to miss sprint timeline"
        })

    # 3️⃣ Dependency-driven replanning
    dependency_map = risks.get("blocked_dependencies_map", {})
    if dependency_map:
        decisions.append({
            "action": "Re-sequence dependent stories",
            "details": dependency_map,
            "reason": "Downstream stories depend on blocked work"
        })

    # 4️⃣ Low sprint health → scope adjustment
    if sprint_health < 0.6:
        decisions.append({
            "action": "Reduce sprint scope",
            "reason": f"Sprint health is low ({sprint_health})"
        })

    # 5️⃣ Always propose prioritized backlog
    if prioritized_backlog:
        decisions.append({
            "action": "Proposed sprint backlog order",
            "ordered_issues": prioritized_backlog,
            "reason": "Aligned with PM focus weights"
        })

    state["plan_decisions"] = decisions


    unplanned_risk = state.get("unplanned_work_risk", {})
    relevant_defects = state.get("relevant_defects", [])

    if unplanned_risk.get("risk_level") == "high":
        decisions.append({
            "action": "Reserve sprint capacity",
            "buffer_pct": unplanned_risk["buffer_pct"],
            "reason": "High likelihood of unplanned work",
        })

    if relevant_defects:
        decisions.append({
            "action": "Include defect remediation candidates",
            "defect_ids": [d["id"] for d in relevant_defects],
            "reason": "Historical defects overlap sprint scope",
        })

        aligned_defects = state.get("aligned_defects", [])


    print("[PLANNING AGENT] Decisions:")
    for d in decisions:
        print(" -", d)

    record_decision(
        state,
        agent="planning_agent",
        type="sprint_plan",

        context_facts={
            "sprint_health": sprint_health,
            "risk_categories": list(risks.keys()),
        },

        commitment_action={
            "planning_decisions": decisions,
        },

        rules_fired=[
            "oversized_split",
            "blocked_removal",
            "dependency_resequence",
        ],

        alternatives=["keep_scope_constant"],
        confidence=0.8,
    )

    state["next_agent"] = "llm_story_split"
    return state



In [ ]:
# from otel_decorator import agent_span

@agent_span()

def llm_story_split_agent(state: dict) -> dict:
    oversized = state.get("estimation_flags", [])
    if not oversized:
        return state

    state.setdefault("llm_traces", [])

    prompt = f"""
    You are an agile coach.

    Suggest how to split the following oversized user stories into smaller,
    independently deliverable stories.

    Constraints:
    - Do not change total scope
    - Each resulting story should be <= 5 story points
    - Provide clear acceptance criteria for each split

    Oversized stories:
    {oversized}
    """

    messages = [
        SystemMessage(content="You suggest story splits, not estimates."),
        HumanMessage(content=prompt)
    ]

    response = llm.invoke(messages)

    # Full traceability
    state["llm_traces"].append({
        "agent": "llm_story_split_agent",
        "model": "gpt-4o-mini",
        "prompt": prompt,
        "response": response.content
    })

    state["story_split_suggestions"] = response.content

    print("[LLM STORY SPLIT AGENT] Suggestions generated")
    print(f"state: {state}")

    record_decision(
        state,
        agent="llm_story_split_agent",
        type="generate_story_splits",
        context_facts={"oversized_count": len(oversized)},
        commitment_action={"suggestions": "generated"},
        rules_fired=["story_point_constraint"],
        alternatives=["manual_split"],
        confidence=None,  # intentionally unknown
    )

    return state


In [ ]:
def router(state: dict) -> str:
    if "next_agent" not in state:
        raise RuntimeError(
            f"next_agent not set. Current state keys: {list(state.keys())}"
        )
    return state["next_agent"]


In [ ]:
from langgraph.graph import StateGraph
from langgraph.checkpoint.memory import MemorySaver


graph = StateGraph(dict)

graph.add_node("jira", jira_data_agent)
graph.add_node("estimation", estimation_agent)

graph.add_node(
    "defect_alignment",
    defect_backlog_semantic_alignment_agent
)

graph.add_node(
    "human_alignment_review",
    human_alignment_review_agent
)

graph.add_node(
    "unplanned_work_prediction",
    unplanned_work_prediction_agent
)

graph.add_node("sprint_health", sprint_health_agent)
graph.add_node("prioritization", prioritization_agent)
graph.add_node("planning", planning_agent)
graph.add_node("llm_story_split", llm_story_split_agent)

graph.set_entry_point("jira")

graph.add_conditional_edges(
    "jira",
    router,
    {"estimation": "estimation"}
)

graph.add_conditional_edges(
    "estimation",
    router,
    {"defect_alignment": "defect_alignment"}
)

graph.add_conditional_edges(
    "defect_alignment",
    router,
    {
        "human_alignment_review": "human_alignment_review",
        "unplanned_work_prediction": "unplanned_work_prediction",
    }
)

graph.add_conditional_edges(
    "human_alignment_review",
    router,
    {"unplanned_work_prediction": "unplanned_work_prediction"}
)

graph.add_conditional_edges(
    "unplanned_work_prediction",
    router,
    {"sprint_health": "sprint_health"}
)

graph.add_conditional_edges(
    "sprint_health",
    router,
    {"prioritization": "prioritization"}
)

graph.add_conditional_edges(
    "prioritization",
    router,
    {"planning": "planning"}
)

graph.add_conditional_edges(
    "planning",
    router,
    {"llm_story_split": "llm_story_split"}
)



checkpointer = MemorySaver()

# app = graph.compile(checkpointer=checkpointer)
app = graph.compile()



In [ ]:
initial_state = {
    "sprint_goal": "Improve payment reliability",
    "team_velocity": 13,
    "new_members": 1,  # can adjust
    "focus_weights": {"feature":0.5,"enhancement":0.3,"bugfix":0.2},
    # "issues", "risks", etc will be initialized by agents
}

In [ ]:
from opentelemetry import trace, context

tracer = trace.get_tracer("pm-agent")

with tracer.start_as_current_span("langgraph.run") as root_span:
    token = context.attach(trace.set_span_in_context(root_span))
    try:
        # --- Run the workflow directly ---
        final_state = app.invoke(initial_state)

        # --- Print final state ---
        print("\n[FINAL STATE] Complete state:")
        for k, v in final_state.items():
            print(f"{k}: {v}")

    finally:
        context.detach(token)


[JIRA DATA AGENT] Issues loaded:
 - PAY-101, blocked=False
 - 13
 - AUTH-44, blocked=False
 - 5
 - BUG-17, blocked=False
 - 3
 - API-88, blocked=True
 - 21
 - OPS-9, blocked=False
 - 8
[ESTIMATION AGENT] [{'issue': 'API-88', 'points': 21, 'velocity': 15, 'recommendation': 'Split user story'}]
DEBUG(defect_alignment): current_issues at start: [JiraIssue(key='PAY-101', story_points=13, status='To Do', type='feature', blocked=False, dependencies=[], days_delayed=0), JiraIssue(key='AUTH-44', story_points=5, status='In Progress', type='enhancement', blocked=False, dependencies=[], days_delayed=0), JiraIssue(key='BUG-17', story_points=3, status='To Do', type='bugfix', blocked=False, dependencies=[], days_delayed=0), JiraIssue(key='API-88', story_points=21, status='To Do', type='feature', blocked=True, dependencies=[], days_delayed=12), JiraIssue(key='OPS-9', story_points=8, status='To Do', type='enhancement', blocked=False, dependencies=['API-88'], days_delayed=0)]

Current historical defect